# 🔄 RERANKER EXPERIMENTS - 3 Cross-Encoders on 6 Datasets

Test cross-encoder rerankers: BGE, Jina, Qwen

**Note**: Rerankers are slower but more accurate than bi-encoders!

## Setup: Clone from GitHub

In [ ]:
# 1. Mount Drive (sonuçlar Drive'a kaydedilecek)
from google.colab import drive
drive.mount('/content/drive')

# 2. GitHub'dan clone et
!git clone https://github.com/EngindalgaMaku/zeroshot-text-classification-benchmark-.git
%cd zeroshot-text-classification-benchmark-

# 3. Sonuçları Drive'a kaydetmek için symlink oluştur
import os
drive_results = '/content/drive/MyDrive/zeroshot_reranker_results'
os.makedirs(drive_results, exist_ok=True)

if os.path.exists('results'):
    !rm -rf results
!ln -s {drive_results} results
print(f"✅ Results will be saved to: {drive_results}")

# 4. Paketleri yükle
!pip install -q -r requirements.txt
print("✅ Setup complete!")

In [ ]:
import torch
print(f"GPU: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"{torch.cuda.get_device_name(0)} - {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 📊 6 Datasets × 2 Rerankers = 12 Experiments!

### Datasets:
1. **AG News**: 4 classes
2. **DBpedia-14**: 14 classes
3. **Yahoo Answers**: 10 classes
4. **Banking77**: 77 classes
5. **Twitter Financial**: 3 classes
6. **20 Newsgroups**: 20 classes

### 2 Rerankers (Cross-Encoders):
1. **BGE Reranker v2-m3** (568M) - Multilingual
2. **Qwen3 Reranker** (600M) - Multilingual (converted for sentence-transformers)

**Note**: Jina Reranker v2 excluded due to transformers compatibility issues

In [ ]:
import yaml
from pathlib import Path

# 6 datasets
datasets = [
    ("ag_news", "text", "label", 1000),
    ("dbpedia_14", "content", "label", 1000),
    ("yahoo_answers_topics", "best_answer", "topic", 1000),
    ("banking77", "text", "label", None),
    ("zeroshot/twitter-financial-news-sentiment", "text", "label", 500),
    ("SetFit/20_newsgroups", "text", "label", 1000),
]

# 2 rerankers (Jina excluded due to transformers compatibility)
rerankers = [
    ("BAAI/bge-reranker-v2-m3", "bge_reranker"),
    ("tomaarsen/Qwen3-Reranker-0.6B-seq-cls", "qwen_reranker"),
]

Path("experiments/reranker").mkdir(parents=True, exist_ok=True)

for ds_name, text_col, label_col, max_samples in datasets:
    for model_name, model_short in rerankers:
        ds_clean = ds_name.replace("/", "_").replace("-", "_")
        exp_name = f"{ds_clean}_{model_short}"
        
        split = "validation" if "twitter" in ds_name or "financial" in ds_name else "test"
        
        config = {
            "experiment_name": exp_name,
            "dataset": {
                "name": ds_name,
                "split": split,
                "text_column": text_col,
                "label_column": label_col,
                "max_samples": max_samples
            },
            "task": {
                "type": "zero_shot_classification",
                "label_mode": "description",
                "language": "en"
            },
            "models": {
                "reranker": {"provider": "hf", "name": model_name}
            },
            "pipeline": {"mode": "reranker"},
            "evaluation": {"metrics": ["accuracy", "macro_f1", "per_class_f1"]},
            "output": {"save_predictions": True, "save_metrics": True, "output_dir": "results/raw"}
        }
        with open(f"experiments/reranker/{exp_name}.yaml", "w") as f:
            yaml.dump(config, f)
        print(f"✅ {exp_name}")

print(f"\n📊 Total: {len(datasets) * len(rerankers)} experiments created! (6 datasets × 2 rerankers = 12)")

In [ ]:
# ⚙️ SETTINGS
SKIP_EXISTING = True

import glob

configs = sorted(glob.glob("experiments/reranker/*.yaml"))
print(f"Found {len(configs)} experiments")
if SKIP_EXISTING:
    print("Mode: --skip-existing\n")
else:
    print("Mode: OVERWRITE\n")

results = []
for i, config in enumerate(configs, 1):
    exp_name = config.split("/")[-1].replace(".yaml", "")
    print(f"\n{'='*70}")
    print(f"[{i}/{len(configs)}] {exp_name}")
    print(f"{'='*70}")
    try:
        cmd = f"python main.py --config {config}"
        if SKIP_EXISTING:
            cmd += " --skip-existing"
        !{cmd}
        results.append((exp_name, "✅"))
    except Exception as e:
        print(f"ERROR: {e}")
        results.append((exp_name, "❌"))

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
success = sum(1 for _, s in results if s == "✅")
print(f"Success: {success}/{len(results)}\n")
for exp, status in results:
    print(f"{status} {exp}")

In [ ]:
# Load and organize results
import json
import pandas as pd
from pathlib import Path

files = list(Path("results/raw").glob("*_metrics.json"))
data = []

for f in files:
    if any(ds in f.name for ds in ["ag_news", "dbpedia", "yahoo", "banking", "twitter", "financial", "20_newsgroups", "SetFit"]):
        if "reranker" not in f.name:
            continue
        
        with open(f) as fp:
            m = json.load(fp)
        
        exp_name = m.get("experiment_name", f.stem)
        
        # Dataset mapping
        if "ag_news" in exp_name:
            dataset = "AG News"
        elif "dbpedia" in exp_name:
            dataset = "DBpedia-14"
        elif "yahoo" in exp_name:
            dataset = "Yahoo Answers"
        elif "banking" in exp_name:
            dataset = "Banking77"
        elif "twitter" in exp_name or "financial" in exp_name:
            dataset = "Twitter Financial"
        elif "20_newsgroups" in exp_name or "SetFit" in exp_name:
            dataset = "20 Newsgroups"
        else:
            continue
        
        # Model mapping
        if "bge_reranker" in exp_name:
            model = "BGE Reranker"
        elif "jina_reranker" in exp_name:
            model = "Jina Reranker"
        elif "qwen_reranker" in exp_name:
            model = "Qwen Reranker"
        else:
            model = "Unknown"
        
        data.append({
            "dataset": dataset,
            "model": model,
            "samples": m["num_samples"],
            "num_classes": len(set([v for v in m["classification_report"].keys() if v.isdigit()])),
            "accuracy": m["accuracy"] * 100,
            "macro_f1": m["macro_f1"] * 100,
        })

df = pd.DataFrame(data)
print("\n" + "="*70)
print("RERANKER RESULTS (2 MODELS × 6 DATASETS)")
print("="*70)
print(df.to_string(index=False))
df.to_csv("results/RERANKER_RESULTS.csv", index=False)
print("\n✅ results/RERANKER_RESULTS.csv")

In [ ]:
# Pivot tables
pivot_acc = df.pivot(index="model", columns="dataset", values="accuracy")
pivot_f1 = df.pivot(index="model", columns="dataset", values="macro_f1")

print("\n" + "="*70)
print("ACCURACY (%)")
print("="*70)
print(pivot_acc.to_string())

print("\n" + "="*70)
print("MACRO F1 (%)")
print("="*70)
print(pivot_f1.to_string())

In [ ]:
# Visualize
import matplotlib.pyplot as plt
import seaborn as sns

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

# F1 scores by dataset
pivot_f1.plot(kind='bar', ax=ax1)
ax1.set_ylabel("Macro F1 (%)")
ax1.set_title("Reranker Performance Across 6 Datasets (3 Models)")
ax1.legend(title="Dataset", bbox_to_anchor=(1.05, 1), loc='upper left')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', alpha=0.3)

# Heatmap
sns.heatmap(pivot_f1, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax2, cbar_kws={'label': 'Macro F1 (%)'})
ax2.set_title("F1 Score Heatmap (Rerankers)")
ax2.set_xlabel("Dataset")
ax2.set_ylabel("Model")

plt.tight_layout()
plt.savefig("results/plots/reranker_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print("✅ results/plots/reranker_comparison.png")

In [ ]:
# Average performance
avg_perf = df.groupby("model")[["accuracy", "macro_f1"]].mean().sort_values("macro_f1", ascending=False)
print("\n" + "="*70)
print("AVERAGE PERFORMANCE ACROSS ALL 6 DATASETS")
print("="*70)
print(avg_perf.to_string())
print("\n🏆 Best reranker:", avg_perf.index[0])

## ✅ DONE!

### Key Findings:
- **12 experiments** completed (2 rerankers × 6 datasets)
- **Cross-encoders** are more accurate but slower than bi-encoders
- **Best reranker** identified

### Comparison with Bi-encoders:
- Rerankers: Higher accuracy, slower inference
- Bi-encoders: Faster inference, lower accuracy
- Use case: Rerankers for high-accuracy, bi-encoders for speed